In [1]:
import backtrader as bt
import pandas as pd
import numpy as np
import datetime
import yfinance as yf
from copy import deepcopy

# 小试一下

In [2]:
# 实例化 cerebro
cerebro = bt.Cerebro()
# 打印初始资金
print('Starting Portfolio Value: %.2f' % cerebro.broker.getvalue())
# 启动回测
cerebro.run()
# 打印回测完成后的资金
print('Final Portfolio Value: %.2f' % cerebro.broker.getvalue())

Starting Portfolio Value: 10000.00
Final Portfolio Value: 10000.00


# 一、数据准备

## 1.1 下载日度行情数据

Backtrader 默认情况下要求输入的 7 个字段： 'datetime' 、'open'、'high'、'low'、'close'、'volume'、'openinterest'，外加一个 'sec_code' 股票代码字段。

In [6]:
# Download historical stock daily price from yahoo finance
def download_and_save_csv(ticker="NVDA", start="2024-01-01", end="2025-01-01", filename="yh_data.csv"):
    # Download data
    stock_data = yf.download(ticker, start=start, end=end, interval='1d')

    # Check for multi-index columns and flatten them
    if isinstance(stock_data.columns, pd.MultiIndex):
        stock_data.columns = [col[0] for col in stock_data.columns]  # Take the first level ('Close', 'Open', etc.)

    stock_data.reset_index(inplace=True)  # Ensure Date is a regular column
    stock_data['sec_code'] = ticker  # Add security code (ticker)
    stock_data['openinterest'] = 0  # Set open interest to 0 (not used in equities)
    
    # Rename columns to match backtrader expected format
    stock_data.rename(columns={'Date': 'datetime', 'Open': 'open', 'High': 'high', 'Low': 'low', 
                               'Close': 'close', 'Volume': 'volume'}, inplace=True)
    
    # Select columns in required order
    stock_data = stock_data[['datetime', 'sec_code', 'open', 'high', 'low', 'close', 'volume', 'openinterest']]
    
    # Save to CSV
    #stock_data.to_csv(filename, index=False)
    #print(f"Data saved to {filename}")

    return stock_data

In [35]:
# Download NVDA stock daily price data
ticker = 'NVDA'
start='2022-01-01'
end='2025-01-01'
yh_daily_price = download_and_save_csv(ticker, start, end)
# 以 datetime 为 index，类型为 datetime 或 date 类型，Datafeeds 默认情况下是将 index 匹配给 datetime 字段；
yh_daily_price = yh_daily_price.set_index(['datetime'])
yh_daily_price

[*********************100%***********************]  1 of 1 completed


,sec_code,open,high,low,close,volume,openinterest
datetime,,,,,,,
2022-01-03,NVDA,29.765494,30.660006,29.735543,30.070986,391547000,0
2022-01-04,NVDA,30.226730,30.417413,28.301932,29.241369,527154000,0
2022-01-05,NVDA,28.900935,29.367160,27.487287,27.558168,498064000,0
2022-01-06,NVDA,27.594107,28.390783,27.020063,28.131214,454186000,0
2022-01-07,NVDA,28.094276,28.374810,27.012074,27.201759,409939000,0
...,...,...,...,...,...,...,...
2024-12-24,NVDA,140.000000,141.899994,138.649994,140.220001,105157000,0
2024-12-26,NVDA,139.699997,140.850006,137.729996,139.929993,116205600,0
2024-12-27,NVDA,138.550003,139.020004,134.710007,137.009995,170582600,0


In [36]:
# Define a custom data feed class for Backtrader
class CustomPandasData(bt.feeds.PandasData):
    params = (
        ('datetime', None),
        ('sec_code', 'sec_code'), 
        ('open', 'Open'),
        ('high', 'High'),
        ('low', 'Low'),
        ('close', 'Close'),
        ('volume', 'Volume'),
        ('openinterest', -1),    
    )

# Add the data feed to cerebro
data_feed = CustomPandasData(dataname=yh_daily_price)
data_feed

## 1.2 读取调仓信息表

表内数据说明：

+ trade_date： 调仓期（每月最后一个交易日）;

+ sec_code：持仓成分股；

+ weight：持仓权重。

In [37]:
trade_info = pd.read_csv("./data/trade_info_yahoo.csv", parse_dates=['trade_date'])
trade_info


,trade_date,sec_code,weight
0,2022-01-31,NVDA,0.33
1,2023-01-31,NVDA,0.33
2,2024-01-31,NVDA,0.30


# 二、 选股回测

 选股策略：定期按持仓权重调仓 。

In [38]:
# 回测策略
class TestStrategy(bt.Strategy):
    params = (
        ('buy_stocks', None), # 传入各个调仓日的股票列表和相应的权重
    )
    def log(self, txt, dt=None):
        ''' Logging function fot this strategy'''
        dt = dt or self.datas[0].datetime.date(0)
        print('{}, {}'.format(dt.isoformat(), txt))

    def __init__(self):
         # 读取调仓日期，即每月的最后一个交易日，回测时，会在这一天下单，然后在下一个交易日，以开盘价买入
        self.trade_dates = pd.to_datetime(self.p.buy_stocks['trade_date'].unique()).date.tolist()
        self.buy_stock = self.p.buy_stocks # 保留调仓信息
        self.order_list = []  # 记录以往订单，在调仓日要全部取消未成交的订单
        self.buy_stocks_pre = [] # 记录上一期持仓
    
    def next(self):
        # 获取当前的回测时间点
        dt = self.datas[0].datetime.date(0)
        # 打印当前时刻的总资产
        self.log('当前总资产 %.2f' %(self.broker.getvalue()))
        # 如果是调仓日，则进行调仓操作
        if dt in self.trade_dates:
            print("--------------{} 为调仓日----------".format(dt))
            #取消之前所下的没成交也未到期的订单
            if len(self.order_list) > 0:
                print("--------------- 撤销未完成的订单 -----------------")
                for od in self.order_list:
                    # 如果订单未完成，则撤销订单
                    self.cancel(od) 
                 #重置订单列表
                self.order_list = [] 
                
            # 提取当前调仓日的持仓列表            
            buy_stocks_data = self.buy_stock.query(f"trade_date=='{dt}'")
            long_list = buy_stocks_data['sec_code'].tolist()
            print('long_list', long_list)  # 打印持仓列表
            
            # 对现有持仓中，调仓后不再继续持有的股票进行卖出平仓
            sell_stock = [i for i in self.buy_stocks_pre if i not in long_list]
            print('sell_stock', sell_stock)
            if len(sell_stock) > 0:
                print("-----------对不再持有的股票进行平仓--------------")
                for stock in sell_stock:
                    data = self.getdatabyname(stock)
                    if self.getposition(data).size > 0 :
                        od = self.close(data=data)  
                        self.order_list.append(od) # 记录卖出订单

            # 买入此次调仓的股票：多退少补原则
            print("-----------买入此次调仓期的股票--------------")
            for stock in long_list:
                w = buy_stocks_data.query(f"sec_code=='{stock}'")['weight'].iloc[0] # 提取持仓权重
                data = self.getdatabyname(stock)
                order = self.order_target_percent(data=data, target=w*0.95) # 为减少可用资金不足的情况，留 5% 的现金做备用
                self.order_list.append(order)
                
            self.buy_stocks_pre = long_list  # 保存此次调仓的股票列表
        
    #订单日志    
    def notify_order(self, order):
        # 未被处理的订单
        if order.status in [order.Submitted, order.Accepted]:
            return
        # 已被处理的订单
        if order.status in [order.Completed, order.Canceled, order.Margin]:
            if order.isbuy():
                self.log(
                    'BUY EXECUTED, ref:%.0f，Price: %.2f, Cost: %.2f, Comm %.2f, Size: %.2f, Stock: %s' %
                    (order.ref,
                     order.executed.price,
                     order.executed.value,
                     order.executed.comm,
                     order.executed.size,
                     order.data._name))
            else:  # Sell
                self.log('SELL EXECUTED, ref:%.0f, Price: %.2f, Cost: %.2f, Comm %.2f, Size: %.2f, Stock: %s' %
                        (order.ref,
                         order.executed.price,
                         order.executed.value,
                         order.executed.comm,
                         order.executed.size,
                         order.data._name))

In [39]:
# 实例化大脑
cerebro_ = bt.Cerebro() 

# 按股票代码，依次循环传入数据
for stock in yh_daily_price['sec_code'].unique():
    # 日期对齐
    data = pd.DataFrame(index=yh_daily_price.index.unique())
    df = yh_daily_price.query(f"sec_code=='{stock}'")[['open','high','low','close','volume','openinterest']]
    data_ = pd.merge(data, df, left_index=True, right_index=True, how='left')
    data_.loc[:,['volume','openinterest']] = data_.loc[:,['volume','openinterest']].fillna(0)
    data_.loc[:,['open','high','low','close']] = data_.loc[:,['open','high','low','close']].ffill()
    data_.loc[:,['open','high','low','close']] = data_.loc[:,['open','high','low','close']].fillna(0)
    datafeed = bt.feeds.PandasData(dataname=data_, fromdate=datetime.datetime(2022,1,1), todate=datetime.datetime(2025,1,1))
    cerebro_.adddata(datafeed, name=stock)
    #print(f"{stock} Done !") 
print(f"Load daily price data done!")

Load daily price data done!


In [40]:
cerebro = deepcopy(cerebro_)  # 深度复制已经导入数据的 cerebro_，避免重复导入数据 
# 初始资金 100,000,000    
cerebro.broker.setcash(100000000.0) 
# 佣金，双边各 0.0003
cerebro.broker.setcommission(commission=0.0003) 
# 滑点：双边各 0.0001
cerebro.broker.set_slippage_perc(perc=0.0001) 
# 添加策略
cerebro.addstrategy(TestStrategy, buy_stocks=trade_info) # 通过修改参数 buy_stocks ，使用同一策略回测不同的持仓列表
# 添加分析器
cerebro.addanalyzer(bt.analyzers.TimeReturn, _name='pnl') # 返回收益率时序数据
cerebro.addanalyzer(bt.analyzers.AnnualReturn, _name='_AnnualReturn')
cerebro.addanalyzer(bt.analyzers.SharpeRatio, riskfreerate=0.003, annualize=True, _name='_SharpeRatio')
cerebro.addanalyzer(bt.analyzers.DrawDown, _name='_DrawDown')
# 添加观测器
cerebro.addobserver(bt.observers.Value)  # 查看账户资产变动

# 启动回测
result = cerebro.run()
cerebro.plot()  # 绘制资产变动曲线

2023-01-03, 当前总资产 100000000.00
2023-01-04, 当前总资产 100000000.00
2023-01-05, 当前总资产 100000000.00
2023-01-06, 当前总资产 100000000.00
2023-01-09, 当前总资产 100000000.00
2023-01-10, 当前总资产 100000000.00
2023-01-11, 当前总资产 100000000.00
2023-01-12, 当前总资产 100000000.00
2023-01-13, 当前总资产 100000000.00
2023-01-17, 当前总资产 100000000.00
2023-01-18, 当前总资产 100000000.00
2023-01-19, 当前总资产 100000000.00
2023-01-20, 当前总资产 100000000.00
2023-01-23, 当前总资产 100000000.00
2023-01-24, 当前总资产 100000000.00
2023-01-25, 当前总资产 100000000.00
2023-01-26, 当前总资产 100000000.00
2023-01-27, 当前总资产 100000000.00
2023-01-30, 当前总资产 100000000.00
2023-01-31, 当前总资产 100000000.00
--------------2023-01-31 为调仓日----------
long_list ['NVDA']
sell_stock []
-----------买入此次调仓期的股票--------------
2023-02-01, BUY EXECUTED, ref:3，Price: 19.68, Cost: 31600259.91, Comm 9480.08, Size: 1605814.00, Stock: NVDA
2023-02-01, 当前总资产 101996379.31
2023-02-02, 当前总资产 103225536.62
2023-02-03, 当前总资产 102248310.74
2023-02-06, 当前总资产 102230656.49
2023-02-07, 当前总资产 103970090.26
2023-02

C:\Users\Jacky\AppData\Local\Temp\ipykernel_6648\432829533.py:36: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  buy_stocks_data = self.buy_stock.query(f"trade_date=='{dt}'")
C:\Users\Jacky\AppData\Local\Temp\ipykernel_6648\432829533.py:36: FutureWarning: The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.
  buy_stocks_data = self.buy_stock.query(f"trade_date=='{dt}'")


<IPython.core.display.Javascript object>

[[<Figure size 640x480 with 5 Axes>]]

In [34]:
strat = result[0]
print("--------------- AnnualReturn -----------------")
print(strat.analyzers._AnnualReturn.get_analysis())
print("--------------- SharpeRatio -----------------")
print(strat.analyzers._SharpeRatio.get_analysis())
print("--------------- DrawDown -----------------")
print(strat.analyzers._DrawDown.get_analysis())

--------------- AnnualReturn -----------------
OrderedDict([(2023, 0.47890843465049393), (2024, 0.5145630045031997)])
--------------- SharpeRatio -----------------
OrderedDict([('sharperatio', 27.6955084084055)])
--------------- DrawDown -----------------
AutoOrderedDict([('len', 36), ('drawdown', 4.7995862703541725), ('moneydown', 11292590.94178772), ('max', AutoOrderedDict([('len', 80), ('drawdown', 12.6228328343306), ('moneydown', 28397959.03817749)]))])
